# Área + Linha Categórico — Casos

Para as três variáveis categóricas de Casos (Faixa Etária, Escolaridade
e Raça/Cor), um único gráfico combina a **Quantidade** (valor
absoluto) por categoria como **linha** (eixo Y à esquerda) com a
**Taxa** por categoria como **linha tracejada** (mesma cor da linha de
quantidade correspondente, eixo Y à direita) -- as duas sobrepostas no
mesmo eixo X (Ano).

Nenhuma linha é empilhada: cada categoria parte do seu próprio valor
real, direto comparável com as demais no mesmo eixo -- ao contrário de
uma área empilhada, em que só a ESPESSURA da fatia representa o valor
da categoria (o topo da fatia é a soma acumulada de tudo que está
embaixo dela, não o valor da própria categoria), fácil de ler errado.

TAXA e QUANTIDADE estaduais já vêm prontas do CSV agregado por ano
(`tabela_regional_agregada_por_ano.csv`, gerado por
`calculo_taxa.ipynb`) -- nenhuma agregação é recalculada aqui, só
recortada (`preparar_categorica_estadual_por_ano`) e pivotada para o
gráfico.

In [1]:
import sys
from pathlib import Path


def _localizar_pasta(candidatos, arquivo_marcador):
    pasta = next((p for p in candidatos if (p / arquivo_marcador).exists()), None)
    if pasta is None:
        raise ModuleNotFoundError(
            f"Não encontrei '{arquivo_marcador}' em nenhum destes caminhos:\n"
            + "\n".join(f"  - {p}" for p in candidatos)
        )
    return pasta

_candidatos_pasta_eda = [
    Path.cwd(),
    Path.cwd().parent / "Análise Exploratória de Dados",
    Path("/home/anna/tcc/Mineração de Dados/Análise Exploratória de Dados"),
    Path.home() / "tcc" / "Mineração de Dados" / "Análise Exploratória de Dados",
]
pasta_eda = _localizar_pasta(_candidatos_pasta_eda, "utilitarios_graficos_populacionais.py")
sys.path.append(str(pasta_eda))

from caminho_utilitarios import preparar_sys_path
preparar_sys_path()

PosixPath('/home/anna/tcc/Seleção e Transformação/População')

In [2]:
from utilitarios_comuns import (
    PASTA_POPULACAO,
    ESPECIFICACOES_VARIAVEIS_CATEGORICAS,
    SUFIXO_VARIAVEL_ESTADUAL,
    NOME_ARQUIVO_AGREGADO_CATEGORICO_POR_ANO,
    registrar_log,
)
from utilitarios_graficos_populacionais import (
    carregar_taxa_por_regional,
    preparar_categorica_estadual_por_ano,
    ordenar_categorias,
    gerar_paleta_cores,
    montar_grafico_linha_dupla,
    remover_anos_sem_dado,
    salvar_figura,
)

### Configuração

In [3]:
PASTA_TAXAS = PASTA_POPULACAO / "Taxa"
PASTA_SAIDA_GRAFICOS = Path.cwd() / "Gráficos" / "Linha Dupla Categórico"

SALVAR_FIGURAS = False

ANO_INICIAL_EXIBIDO = None
ANO_FINAL_EXIBIDO = None

LARGURA_FIGURA = 1100
ALTURA_FIGURA = 950

LISTA_VARIAVEIS_CATEGORICAS = [
    {
        "NOME": nome,
        "SUFIXO": SUFIXO_VARIAVEL_ESTADUAL[nome],
        "ANO_INICIAL": ANO_INICIAL_EXIBIDO or especificacao["ANO_INICIAL"],
        "ANO_FINAL": ANO_FINAL_EXIBIDO or especificacao["ANO_FINAL"],
    }
    for nome, especificacao in ESPECIFICACOES_VARIAVEIS_CATEGORICAS.items()
]

tabela_agregada_estadual = carregar_taxa_por_regional(PASTA_TAXAS / NOME_ARQUIVO_AGREGADO_CATEGORICO_POR_ANO)

### Gráfico — Taxa (linha) e Quantidade (área empilhada) por Categoria, sobrepostos

In [4]:
for configuracao in LISTA_VARIAVEIS_CATEGORICAS:
    nome_da_variavel = configuracao["NOME"]
    registrar_log(f"===== INÍCIO: {nome_da_variavel.upper()} =====")

    tabela_pivotada_taxa, tabela_pivotada_quantidade = preparar_categorica_estadual_por_ano(
        tabela_agregada_estadual, configuracao["SUFIXO"],
        configuracao["ANO_INICIAL"], configuracao["ANO_FINAL"],
    )

    # Remove do eixo X anos sem NENHUM dado em nenhuma categoria (nem na
    # taxa, nem na quantidade) -- sem isso, um ano assim vira um ponto
    # vazio/zerado no gráfico. Categorias isoladas ausentes num ano que
    # já tem outras categorias com dado continuam virando 0 na área/linha,
    # só o ano inteiramente vazio é removido -- por isso o fillna(0) só
    # acontece depois do filtro.
    tabela_pivotada_taxa, tabela_pivotada_quantidade = remover_anos_sem_dado(
        tabela_pivotada_taxa, tabela_pivotada_quantidade
    )
    tabela_pivotada_taxa = tabela_pivotada_taxa.fillna(0)
    tabela_pivotada_quantidade = tabela_pivotada_quantidade.fillna(0)

    categorias_ordenadas = ordenar_categorias(tabela_pivotada_taxa.columns.tolist())
    paleta_de_cores = gerar_paleta_cores(categorias_ordenadas)

    figura_combinada = montar_grafico_linha_dupla(
        eixo_x=tabela_pivotada_quantidade.index,
        tabela_pivotada_quantidade=tabela_pivotada_quantidade,
        tabela_pivotada_taxa=tabela_pivotada_taxa,
        titulo=f"{nome_da_variavel} — Valor Absoluto e Taxa por Categoria, Evolução Anual em MG",
        paleta_cores=paleta_de_cores, titulo_eixo_x="Ano", ordem_categorias=categorias_ordenadas,
        largura=LARGURA_FIGURA, altura=ALTURA_FIGURA, tickangle_eixo_x=-45, eixo_x_ano_continuo=True,
    )
    figura_combinada.show()

    if SALVAR_FIGURAS:
        nome_arquivo_base = nome_da_variavel.lower().replace(" ", "_")
        salvar_figura(figura_combinada, f"{nome_arquivo_base}_linha_dupla", PASTA_SAIDA_GRAFICOS)

    registrar_log(f"===== FIM: {nome_da_variavel.upper()} =====\n")

[LOG] ===== INÍCIO: CASOS POR ESCOLARIDADE =====


[LOG] ===== FIM: CASOS POR ESCOLARIDADE =====

[LOG] ===== INÍCIO: CASOS POR RAÇA E COR =====


[LOG] ===== FIM: CASOS POR RAÇA E COR =====

[LOG] ===== INÍCIO: CASOS POR FAIXA ETÁRIA =====


[LOG] ===== FIM: CASOS POR FAIXA ETÁRIA =====

